###### Step 6: 光流与特征跟踪

这一节直接使用：

- `LK_optical_flow`, `HS_optical_flow`
- `flow_to_image`
- `Feature_tracker_sd`

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

cwd = Path.cwd().resolve()
repo_root = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / 'tianmoucv').exists() and (candidate / 'tianmoucv_example').exists():
        repo_root = candidate
        break
if repo_root is None:
    raise RuntimeError('找不到 tianmoucv 仓库根目录，请从仓库内打开本 Notebook。')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import matplotlib.pyplot as plt
import torch


In [ ]:
import torch.nn.functional as F
from tianmoucv.data import TianmoucDataReader
from tianmoucv.isp import SD2XY
from tianmoucv.proc.opticalflow import HS_optical_flow, LK_optical_flow, flow_to_image
from tianmoucv.proc.tracking import Feature_tracker_sd
from tianmoucv.utils import ensure_tutorial_dataset

info = ensure_tutorial_dataset()
dataset = TianmoucDataReader(str(info.sample_dir), N=1, print_info=False)
sample = dataset[5]
rawdiff = sample['rawDiff'].float() 

## 1. 用一段 AOP 窗口估计经典光流

In [ ]:
window = rawdiff[:, 8:16]
sd_acc = None
td_acc = torch.zeros_like(window[0, 0])
for t in range(window.shape[1]):
    ix, iy = window[1:, t]
    cur_sd = torch.stack([ix, iy], dim=0)
    sd_acc = cur_sd if sd_acc is None else sd_acc + cur_sd
    td_acc = td_acc + window[0, t]

sd_acc = sd_acc / window.shape[1]
td_acc = (td_acc / window.shape[1]).unsqueeze(0).unsqueeze(0)
td_acc = F.interpolate(td_acc, size=sd_acc.shape[1:], mode='bilinear', align_corners=False).squeeze(0)

flow_lk = LK_optical_flow(sd_acc, td_acc, win=5, stride=1, mask=None, ifInterploted=False)
flow_hs = HS_optical_flow(sd_acc, td_acc, ifInterploted=False, maxIteration=50, scales=3, labmda=30)

flow_lk_rgb = flow_to_image(flow_lk.permute(1, 2, 0).numpy())
flow_hs_rgb = flow_to_image(flow_hs.permute(1, 2, 0).numpy())

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.imshow(sample['F0'].numpy())
plt.title('F0')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(flow_lk_rgb)
plt.title('LK_optical_flow')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(flow_hs_rgb)
plt.title('HS_optical_flow')
plt.axis('off')
plt.tight_layout()
plt.show()

## 2. 基于 SD 的特征跟踪

In [ ]:
tracker = Feature_tracker_sd(d_factor=0.5)#d_factor越小越严格

for i in range(10):
    sample = dataset[i]
    rawdiff = sample['rawDiff'].float() 
    track_vis = None
    for t in range(25):
        ix, iy = SD2XY(rawdiff[1:, t].permute(1, 2, 0))
        sobel = (ix ** 2 + iy ** 2) ** 0.5
        _, track_vis = tracker.update(sobel, viz=True)

plt.figure(figsize=(8, 4))
plt.imshow(track_vis, cmap='gray')
plt.title('Feature_tracker_sd after 12 AOP frames')
plt.axis('off')
plt.show()